## Importing Libraries

In [1]:
from ollama import chat
import glob
from tqdm import tqdm
import os
import json
import re
import unicodedata
from groq import Groq
from difflib import SequenceMatcher

## Setting up files

In [2]:
GENERATION_MODEL = "qwen3:8b" 
GROQ_MODEL = "openai/gpt-oss-120b"

GROQ_KEY = os.getenv("GROQ_API_KEY")
CLIENT = Groq(api_key=GROQ_KEY)

TYPE_LLM = True # True - local, False - groq

FILES_EXTR = glob.glob("../Test_Files/Clinical_trials/clinical-trial_*.txt")
GOLD_FILES = glob.glob("../Test_Files/Clinical_trials/GT-clinical-trial_*.json")
TRIAL_STRUCTURE_FILES = glob.glob("../Test_Files/Clinical_trials/Trial_structure/clinical-trial-structure_e*.txt")

PROMPT_EXTR_FILE = "./prompts/criteria_extraction/criteria-extraction-cohort_prompt.txt"
SYS_PROMPT_EXTR_FILE = "./prompts/criteria_extraction/sys_criteria-extraction-cohort_prompt.txt"

OUTPUT_EXTR_DIR = "./llm-outputs/criteria-extraction/"
OUTPUT_EXTR_FILE = "experiment"

print(f"Found the following files for extraction - {FILES_EXTR}")
print(f"Found the following golden diaries {GOLD_FILES}")
print(f"Found the following trial structure files {TRIAL_STRUCTURE_FILES}")

Found the following files for extraction - ['../Test_Files/Clinical_trials\\clinical-trial_e1.txt', '../Test_Files/Clinical_trials\\clinical-trial_e10.txt', '../Test_Files/Clinical_trials\\clinical-trial_e11.txt', '../Test_Files/Clinical_trials\\clinical-trial_e12.txt', '../Test_Files/Clinical_trials\\clinical-trial_e13.txt', '../Test_Files/Clinical_trials\\clinical-trial_e14.txt', '../Test_Files/Clinical_trials\\clinical-trial_e15.txt', '../Test_Files/Clinical_trials\\clinical-trial_e16.txt', '../Test_Files/Clinical_trials\\clinical-trial_e17.txt', '../Test_Files/Clinical_trials\\clinical-trial_e18.txt', '../Test_Files/Clinical_trials\\clinical-trial_e19.txt', '../Test_Files/Clinical_trials\\clinical-trial_e2.txt', '../Test_Files/Clinical_trials\\clinical-trial_e20.txt', '../Test_Files/Clinical_trials\\clinical-trial_e21.txt', '../Test_Files/Clinical_trials\\clinical-trial_e22.txt', '../Test_Files/Clinical_trials\\clinical-trial_e23.txt', '../Test_Files/Clinical_trials\\clinical-trial

## Pre-processing

In [3]:
def normalize_docs(text):
    text = normalize_text(text)

    inclusion_match = re.search(
        r"(Inclusion Criteria\s*:?\s*)(.*?)(?=Exclusion Criteria\s*:?)",
        text,
        re.IGNORECASE | re.DOTALL,
    )

    exclusion_match = re.search(
        r"(Exclusion Criteria\s*:?\s*)(.*?)(?=\n(?:Study Plan|Study Design|Investigational Product|Control Product|Study Endpoints|Primary Endpoint|Secondary Endpoints|Safety Endpoints|Follow-Up|Statistical Analysis|References)\b|\Z)",
        text,
        re.IGNORECASE | re.DOTALL,
    )

    if inclusion_match and exclusion_match:

        inclusion_text = inclusion_match.group(2).strip()
        exclusion_text = exclusion_match.group(2).strip()

        text = (
            "Inclusion Criteria:\n"
            f"{inclusion_text}\n\n"
            "Exclusion Criteria:\n"
            f"{exclusion_text}"
        )

    return text

def normalize_text(text):
    text = unicodedata.normalize("NFKC", text)
    
    text = re.sub(r"[‐-‒–—]", "-", text)

    text = re.sub(r"[ \t]+", " ", text)

    text = re.sub(r"\r\n?", "\n", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

## Setting up environment

In [4]:
## Setting evironment
def set_env(prompt_file, sys_prompt_file, output_dir):
    with open(prompt_file,"r", encoding="utf-8") as p:
        base_prompt = p.read().strip()
        
    with open(sys_prompt_file, "r", encoding="utf-8") as sp:
        sys_prompt = sp.read().strip()

    os.makedirs(output_dir,exist_ok=True)

    count = 0

    for path in os.listdir(output_dir):
        if os.path.isfile(os.path.join(output_dir, path)):
            count += 1
    
    return base_prompt, sys_prompt, count

base_prompt_extr, sys_prompt_extr, count_extr_exp = set_env(PROMPT_EXTR_FILE, SYS_PROMPT_EXTR_FILE, OUTPUT_EXTR_DIR)

## Criteria Extraction
In this first phase the criteria of a given clinical trial are extracted still in natural language to make the conversion easier

In [ ]:
pbar = tqdm(total=len(FILES_EXTR), desc="Processing trials for criteria extraction")

for file in FILES_EXTR:
    trial_id = file.split("_")[-1].split(".")[0]
    
    print(f"Processing trial {trial_id} for criteria extraction...")
    
    for struc in TRIAL_STRUCTURE_FILES:
        struc_id = struc.split("_")[-1].split(".")[0]
        if struc_id == trial_id:
            with open(struc, "r", encoding="utf-8") as s:
                structure_text = s.read()
                struc_json = json.loads(structure_text)
                cohorts = struc_json.get("cohorts", [])
                cohorts_context = "\n".join(
                    f"- ID: {c['cohort_id']} | Name: {c['name']}"
                    for c in cohorts
                )
                print(f"Cohorts for trial {trial_id}: {cohorts_context}")
                print(f"Found matching trial structure file {struc} for trial {trial_id}")
                print(f"Trial structure content:\n{struc_json}\n")
                has_cohort = 'has_cohorts' in struc_json and struc_json['has_cohorts'] == True
                print(f"Trial {trial_id} has cohort: {has_cohort}")

    with open(file,"r", encoding="utf-8") as f:
        text = f.read()
        
        normalized_text = normalize_docs(text)
        
        print(f"processing file: {file}")
        
        prompt_w_cohort = ""
        
        if has_cohort:
            print(f"Injecting cohort information into prompt for trial {trial_id}")
            prompt_w_cohort = base_prompt_extr.replace("{{COHORTS_CONTEXT}}", cohorts_context)
        else:
            print(f"No cohort information available for trial {trial_id}, using default prompt")
            prompt_w_cohort = base_prompt_extr.replace("{{COHORTS_CONTEXT}}","{No cohorts available for this clinical trial}")

        prompt = base_prompt_extr.replace("{{TRIAL_TEXT}}", normalized_text)
        
        print(f"System prompt for file {file}:\n{sys_prompt_extr}\n")
        print(f"Prompt for file {file}:\n{prompt}\n")
        
        if TYPE_LLM:
            stream = chat(
                model=GENERATION_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": sys_prompt_extr
                    },
                    {
                        "role": "user", 
                        "content": prompt
                        }
                    ],
                stream=True,
                options={"num_ctx": 32000}
                )
            
            llm_output = ""
            for chunk in stream:
                llm_output += chunk["message"]["content"]
                
        elif not TYPE_LLM:
            stream = CLIENT.chat.completions.create(
                model= GROQ_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": sys_prompt_extr
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0
            )
            
            llm_output = stream.choices[0].message.content

        with open(f"{OUTPUT_EXTR_DIR}{OUTPUT_EXTR_FILE}-{count_extr_exp}.txt","a",encoding="utf-8") as o:
            o.write(f"Ouput for file {file}\n")
            o.write(f"{llm_output}\n\n")
            print(f"Saved LLM output on {OUTPUT_EXTR_FILE}-{count_extr_exp}")
            
        
        print("\n")
        
        pbar.update(1)
        
pbar.close()

Processing trials for criteria extraction:   0%|          | 0/30 [00:00<?, ?it/s]

Processing trial e1 for criteria extraction...
Cohorts for trial e1: 
Found matching trial structure file ../Test_Files/Clinical_trials/Trial_structure\clinical-trial-structure_e1.txt for trial e1
Trial structure content:
{'has_cohorts': False, 'cohorts': []}

Trial e1 has cohort: False
processing file: ../Test_Files/Clinical_trials\clinical-trial_e1.txt
No cohort information available for trial e1, using default prompt
System prompt for file ../Test_Files/Clinical_trials\clinical-trial_e1.txt:
You are an assistant responsible for extracting eligibility criteria from a clinical trial.

The trial may have multiple cohorts. Criteria may be general (applying to all cohorts) or specific to one cohort.

Prompt for file ../Test_Files/Clinical_trials\clinical-trial_e1.txt:
Extraction Requirements:

- Extract ALL inclusion and exclusion criteria present in the text.
- Each criterion must be kept exactly as written (no rewriting or interpretation).
- If a criterion contains multiple independent